## Lab - EDA Univariate Analysis: Diving into Amazon UK Product Insights

**Objective**: Explore the product listing dynamics on Amazon UK to extract actionable business insights. By understanding the distribution, central tendencies, and relationships of various product attributes, businesses can make more informed decisions on product positioning, pricing strategies, and inventory management.

**Dataset**: This lab utilizes the [Amazon UK product dataset](https://www.kaggle.com/datasets/asaniczka/uk-optimal-product-price-prediction/)
which provides information on product categories, brands, prices, ratings, and more from from Amazon UK. You'll need to download it to start working with it.

In [243]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("amz_uk_price_prediction_dataset.csv")
df.head()

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


In [244]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2443651 entries, 0 to 2443650
Data columns (total 9 columns):
 #   Column             Dtype  
---  ------             -----  
 0   uid                int64  
 1   asin               object 
 2   title              object 
 3   stars              float64
 4   reviews            int64  
 5   price              float64
 6   isBestSeller       bool   
 7   boughtInLastMonth  int64  
 8   category           object 
dtypes: bool(1), float64(2), int64(3), object(3)
memory usage: 151.5+ MB


### Part 1: Understanding Product Categories

**Business Question**: What are the most popular product categories on Amazon UK, and how do they compare in terms of listing frequency?

1. **Frequency Tables**:
    - Generate a frequency table for the product `category`.
    - Which are the top 5 most listed product categories?

2. **Visualizations**:
    - Display the distribution of products across different categories using a bar chart. *If you face problems understanding the chart, do it for a subset of top categories.*
    - For a subset of top categories, visualize their proportions using a pie chart. Does any category dominate the listings?


In [245]:
tt = df["category"].value_counts()  # Create a frequency table

print("The top 5 most common categories are:")
for i in tt.index[0:5]:     # Print the index of first 5 elements
    print(f"    *{i}")

# Sports & Outdoors dominate the categories with 836265 products, while the second one has 19312

The top 5 most common categories are:
    *Sports & Outdoors
    *Beauty
    *Handmade Clothing, Shoes & Accessories
    *Bath & Body
    *Birthday Gifts


In [246]:
category_counts = tt.reset_index()  # Transform it into a dataframe

px.bar(category_counts, x="category", y="count")    # Ilegible plot. Sports & Outdoors category is so large others do not appear


In [247]:
px.pie(category_counts[1:11], values="count", names="category")     # Top 10, excluding the first one.
# No category dominates the listing, except the first one

### Part 2: Delving into Product Pricing

**Business Question**: How are products priced on Amazon UK, and are there specific price points or ranges that are more common?

1. **Measures of Centrality**:
    - Calculate the mean, median, and mode for the `price` of products.
    - What's the average price point of products listed? How does this compare with the most common price point (mode)?

In [248]:
print(f"The average price is: {df["price"].describe().round(2)["mean"]}")
print(f"The most median price is: {df["price"].describe().round(2)["50%"]}")
print(f"The most common price is: {df["price"].mode().values[0]}")      # Need to use values[0] to extract the values from the series, and return the first one as a float i/o list


The average price is: 89.24
The most median price is: 19.09
The most common price is: 9.99


2. **Measures of Dispersion**:
    - Determine the variance, standard deviation, range, and interquartile range for product `price`.
    - How varied are the product prices? Are there any indicators of a significant spread in prices?

In [249]:
print(f"Variance is: {df["price"].var().__round__(2)}")             # Variance
print(f"Standard Deviation is: {df["price"].std().__round__(2)}")          # Standard deviation
print(f"Range is: {df["price"].max() - df["price"].min()}")         # Range
print(f"Interquartile range is: {df["price"].quantile(0.75) - df["price"].quantile(0.25)}")     # Interquartile range

# It seems that most prices are close to the mode (deducted from the interquartile range), but there are some outliers that affect the variance significantly.
# Without further analysis I can't conclude if STD is large or not.

Variance is: 119445.49
Standard Deviation is: 345.61
Range is: 100000.0
Interquartile range is: 36.0


3. **Visualizations**:
    - Is there a specific price range where most products fall? Plot a histogram to visualize the distribution of product prices. *If its hard to read these diagrams, think why this is, and explain how it could be solved.*.
    - Are there products that are priced significantly higher than the rest? Use a box plot to showcase the spread and potential outliers in product pricing. 

In [250]:
df_price = df["price"].value_counts().reset_index()     # Create a df for the histogram

px.histogram(df_price,     
             x = "price",
             title = "Most prices fall between 25-125 units"
             ).update_xaxes(range=[0, df_price[df_price["count"]>10]["price"].max()])     # Range of X axis starts on 0, ends on the highest price that has been repeated at least 10 times. This way the plot is legible and removes all the unnecesary data 


### Part 3: Unpacking Product Ratings

**Business Question**: How do customers rate products on Amazon UK, and are there any patterns or tendencies in the ratings?

1. **Measures of Centrality**:
    - Calculate the mean, median, and mode for the `rating` of products.
    - How do customers generally rate products? Is there a common trend?


In [251]:
print(f"Mean: {df["stars"].mean()}")
print(f"Median: {df["stars"].median()}")
print(f"Mode: {df["stars"].mode().values[0]}")

# It seems that customers tend to rate the products with 0 stars. My suspicion is that 0 starts actually mean that customer didn't review the product yet

Mean: 2.152836472966065
Median: 0.0
Mode: 0.0


In [252]:
df_reviews = df[df["reviews"]>0]        # Create a filtered df with products that have been reviewd at least once

print(f"Mean: {df_reviews["stars"].mean()}")
print(f"Median: {df_reviews["stars"].median()}")
print(f"Mode: {df_reviews["stars"].mode().values[0]}")

# It seems that once products are reviewd, they have good ratings

Mean: 4.31916076222691
Median: 4.4
Mode: 4.5


2. **Measures of Dispersion**:
    - Determine the variance, standard deviation, and interquartile range for product `rating`.
    - Are the ratings consistent, or is there a wide variation in customer feedback?

In [253]:
print(f"Variance: {df_reviews["stars"].var().__round__(2)}")
print(f"Standard deviation: {df_reviews["stars"].std().__round__(2)}")
print(f"Interquartile range: {df_reviews["stars"].quantile(0.75) - df_reviews["stars"].quantile(0.25)}")

# Ratings tend to be very consistent

Variance: 0.31
Standard deviation: 0.56
Interquartile range: 0.5


3. **Shape of the Distribution**:
    - Calculate the skewness and kurtosis for the `rating` column. 
    - Are the ratings normally distributed, or do they lean towards higher or lower values?

In [254]:
print(f"Skewness: {df_reviews["stars"].skew()}")
print(f"Kurtosis: {df_reviews["stars"].kurtosis()}")

# The prices are skewed to the left, meaning that there are more prices on the right side of the distribution
# As Kurtosis is much larger than 3, it means that the rating distribution has a sharper peak

Skewness: -2.3795681122127945
Kurtosis: 9.78193769066435


4. **Visualizations**:
    - Plot a histogram to visualize the distribution of product ratings. Is there a specific rating that is more common?

In [256]:
px.histogram(df_reviews,
             x = "stars",
             title = "The most common rating is 4.5")